# DP2 light curves on the Rubin Science Platform

This notebook runs on the Rubin Science Platform at [data.lsst.cloud](https://data.lsst.cloud) and requires Rubin data rights. It (i) lists the DP2 tables, (ii) selects DiaObjects with well-sampled light curves, (iii) retrieves their forced photometry on difference images, and (iv) converts the result into the format of the Day 1 Hyrax dataset.

Table and column names follow the DP2 schema browser at [sdm-schemas.lsst.io/dp2.html](https://sdm-schemas.lsst.io/dp2.html). Check them there if a query fails.

DP2 data stays on the Rubin Science Platform. Do not download it to your laptop or commit it to the school repository.

In [ ]:
import numpy as np
import pandas as pd
from lsst.rsp import get_tap_service

service = get_tap_service("tap")
assert service is not None

## 1. List the DP2 tables

In [ ]:
tables = service.search(
    "SELECT table_name, description FROM tap_schema.tables WHERE schema_name = 'dp2'"
).to_table().to_pandas()
tables

## 2. Select DiaObjects with well-sampled light curves

`nDiaSources` counts the difference-image detections of each DiaObject. Start with a small sample and increase `limit` once the queries work.

In [ ]:
min_detections = 20
limit = 1000

query = f"""
SELECT TOP {limit} diaObjectId, ra, dec, nDiaSources
FROM dp2.DiaObject
WHERE nDiaSources >= {min_detections}
"""
objects = service.search(query).to_table().to_pandas()
print(len(objects), "DiaObjects")
objects.head()

## 3. Retrieve forced photometry on difference images

The forced-photometry table is very large, so query it by `diaObjectId` and join to the `Visit` table for the observation time. `psfDiffFlux` is measured on the difference image, so the static host galaxy is removed.

In [ ]:
def light_curves(object_ids):
    ids = ", ".join(str(int(i)) for i in object_ids)
    query = f"""
    SELECT fsodo.diaObjectId, fsodo.band, fsodo.psfDiffFlux, fsodo.psfDiffFluxErr,
           vis.expMidptMJD
    FROM dp2.ForcedSourceOnDiaObject AS fsodo
    JOIN dp2.Visit AS vis ON vis.visit = fsodo.visit
    WHERE fsodo.diaObjectId IN ({ids})
    """
    return service.search(query).to_table().to_pandas()

chunks = [light_curves(objects.diaObjectId[i:i + 100]) for i in range(0, len(objects), 100)]
lc = pd.concat(chunks, ignore_index=True).dropna(subset=["psfDiffFlux", "psfDiffFluxErr"])
print(len(lc), "photometry points")
lc.head()

## 4. Convert to the format of the Day 1 Hyrax dataset

The Day 1 ZTF dataset uses one row per observation with the columns `object_id`, `mjd`, `band`, `flux`, and `flux_err`. Rubin fluxes are in nJy and can be negative in forced photometry, so we keep fluxes and scale them per object instead of converting to magnitudes. For reference, the AB magnitude of a positive flux in nJy is $m = 31.4 - 2.5 \log_{10} f$.

In [ ]:
dp2 = lc.rename(columns={
    "diaObjectId": "object_id",
    "expMidptMJD": "mjd",
    "psfDiffFlux": "flux",
    "psfDiffFluxErr": "flux_err",
})[["object_id", "mjd", "band", "flux", "flux_err"]].sort_values(["object_id", "mjd"])

scale = dp2.groupby("object_id")["flux"].transform(lambda f: np.nanmax(np.abs(f)))
dp2["flux_scaled"] = dp2["flux"] / scale
dp2["flux_err_scaled"] = dp2["flux_err"] / scale

dp2.groupby("band").size()

## 5. Save in your Rubin Science Platform home directory

The file stays on the Rubin Science Platform and is the input of the hackathon.

In [ ]:
from pathlib import Path

out = Path.home() / "rubin-ml-school" / "dp2_light_curves.parquet"
out.parent.mkdir(parents=True, exist_ok=True)
dp2.to_parquet(out, index=False)
print("saved", out)